In [ ]:
# --- make the transformer_pipeline package importable from experiments/ ---
import os, sys
_PKG = os.path.abspath(os.path.join(os.getcwd(), "..", "transformer_pipeline"))
if _PKG not in sys.path:
    sys.path.insert(0, _PKG)

# Stage 2 + 3 - CLR microbiome (x 3 tokenizers)

**The change.** The microbiome is now put into **CLR** (centered log-ratio)
space everywhere, controlled by `DataConfig.microbiome_clr`:

```
clr(x)_i = ln(x_i + eps) - mean_j ln(x_j + eps)      then per-feature z-score
```

- **Stage 3 (Model A input).** Microbiome is the *input* to A -> fed as
  standardized-CLR instead of raw relative abundance. Raw abundances are all
  tiny (~1/170), so the value channel had almost no dynamic range; CLR spreads
  them out.
- **Stage 2 (Model B target).** Microbiome is the *target* of B -> B now
  predicts standardized-CLR with an **identity** head (not softmax), trained
  with MSE. The old softmax-on-simplex target made `val_mse` numerically flat
  (~0.0003) and uninformative; in CLR space MSE/`val_mse` are meaningful again,
  so we can select Model B on `val_mse` like Model A. At imputation time B's
  output is inverse-CLR'd (softmax) back to relative abundance.

Semantics note: after CLR, **0 means "average abundance"**, not "absent" -
an absent microbe maps to a strongly negative value. This mirrors the
standardized metabolome (0 = mean), so both sides are treated symmetrically and
neither input is masked.

**What this notebook does.** Runs the full grid `clr in {off, on}` x
`tokenizer in {scale, add, affine}` for BOTH models, and compares per-feature r
against the ridge ceilings.

> FAIR ridge ceilings (per-feature r, same input/target space as the model):
> A raw 0.12 | **A CLR 0.15** | B raw 0.14 | **B CLR 0.21**
> Compare the CLR transformer to the CLR ridge: the bar is **A 0.15, B 0.21**.
> (The old "A 0.12" ridge saw RAW microbiome input; with CLR input ridge itself
> climbs to 0.15, so most of A's gain is CLR, not the transformer.)

Run from the repo root (folder containing `pipeline/`): Cell > Run All.
Heads up: this is 6 configs x 2 models. Trim `CLR_MODES`/`MODES` if too slow.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import pandas as pd
from pipeline.config import Config
from pipeline.pipeline import ImputationPipeline

EPOCHS    = 60
PERMS     = 99
MODES     = ["scale", "add", "affine"]   # tokenizer fusion
CLR_MODES = [False, True]                # microbiome raw vs CLR

def run(clr, tokenizer):
    cfg = Config()
    cfg.data.microbiome_clr = clr
    cfg.model_a.tokenizer = tokenizer
    cfg.model_b.tokenizer = tokenizer
    cfg.train.max_epochs = EPOCHS
    cfg.train.patience   = EPOCHS        # no early stop -> full curve
    cfg.eval.mantel_permutations = PERMS
    # With CLR, Model B's val_mse is meaningful -> select on it (like A).
    # Without CLR, B's val_mse is flat -> fall back to per_feature_r for B.
    pipe = ImputationPipeline(cfg)
    pipe.prepare_data()
    ta = pipe.train_model_a()
    if not clr:
        cfg.train.selection_metric = "per_feature_r"
    tb = pipe.train_model_b()
    return ta, tb

results = {}
for clr in CLR_MODES:
    for m in MODES:
        tag = ("CLR " if clr else "raw ") + m
        print(f"\n========== microbiome={'CLR' if clr else 'raw'}  tokenizer={m} ==========")
        results[(clr, m)] = run(clr, m)

### Summary: full grid vs the ridge ceilings

In [ ]:
def selected(trainer):
    e = trainer.history.best_epoch
    rec = [r for r in trainer.history.records if r.epoch == e][0]
    return rec.per_feature_r, rec.val_mse

def peak_r(trainer):
    vals = [r.per_feature_r for r in trainer.history.records
            if r.per_feature_r is not None]
    return max(vals) if vals else float("nan")

def min_train_mse(trainer):
    return min(r.train_loss for r in trainer.history.records)

rows = []
for model_name, idx in [("A micro->metab", 0), ("B metab->micro", 1)]:
    for clr in CLR_MODES:
        for m in MODES:
            tr = results[(clr, m)][idx]
            sel_r, sel_mse = selected(tr)
            rows.append({
                "model": model_name,
                "micro": "CLR" if clr else "raw",
                "tokenizer": m,
                "peak_r":         round(peak_r(tr), 4),
                "selected_r":     round(sel_r, 4) if sel_r is not None else None,
                "sel_val_mse":    round(sel_mse, 5),
                "min_train_mse":  round(min_train_mse(tr), 5),
            })

summary = pd.DataFrame(rows)
print("FAIR RIDGE CEILINGS (per-feature r):  A raw 0.12 | A CLR 0.15 | "
      "B raw 0.14 | B CLR 0.21")
print("Bar for the CLR model:  A -> 0.15   B -> 0.21\n")
print(summary.to_string(index=False))

print("\nBest config per model (by peak_r):")
for model_name in summary.model.unique():
    sub = summary[summary.model == model_name]
    b = sub.loc[sub.peak_r.idxmax()]
    print(f"  {model_name}: micro={b.micro} tokenizer={b.tokenizer}  peak_r={b.peak_r}")

### Mantel test for the chosen config

The Mantel statistic is already computed EVERY epoch and printed in the
training log (`Mantel(spearman) r=... p=...`). It is stored on each
`EpochRecord` as `.mantel` (a `MantelResult` with `.statistic`, `.p_value`,
`.n_permutations`, `.method`). The cell below pulls it out for ONE chosen
config so you can read the Mantel score at the *selected* checkpoint plus the
peak over the whole run.

Set `CHOICE = (clr, tokenizer)`. CLR+scale won both models, so that is the
default.

In [ ]:
CHOICE = (True, "scale")   # (microbiome_clr, tokenizer) -> CLR + scale

def mantel_at_selected(trainer):
    e = trainer.history.best_epoch
    rec = [r for r in trainer.history.records if r.epoch == e][0]
    return rec.mantel

def peak_mantel(trainer):
    ms = [r.mantel for r in trainer.history.records if r.mantel is not None]
    return max(ms, key=lambda m: m.statistic) if ms else None

mantel_rows = []
for model_name, idx in [("A micro->metab", 0), ("B metab->micro", 1)]:
    tr = results[CHOICE][idx]
    sel = mantel_at_selected(tr)
    pk  = peak_mantel(tr)
    print(f"{model_name}  (micro={'CLR' if CHOICE[0] else 'raw'}, tok={CHOICE[1]})")
    print(f"   selected-epoch Mantel : {sel}")
    print(f"   peak Mantel over run  : {pk}")
    print()
    if sel is not None:
        mantel_rows.append({
            "model": model_name,
            "sel_mantel_r":  round(sel.statistic, 4),
            "sel_mantel_p":  round(sel.p_value, 4),
            "peak_mantel_r": round(pk.statistic, 4) if pk else None,
        })

if mantel_rows:
    print(pd.DataFrame(mantel_rows).to_string(index=False))

**Caveat (important).** The ridge probe showed the Mantel statistic does
NOT track imputation quality - a strictly better model (higher per-feature r)
can score a *lower* Mantel r, and the sign can even flip. Treat Mantel as a
secondary, geometry-level sanity check (does the predicted sample-distance
structure resemble the truth?), not as the thing you optimise. Selection stays
on `val_mse` / `per_feature_r`.

### Train vs val MSE curves per epoch

For the picked configs (the CLR family by default) this plots **train** MSE
(dashed) and **val** MSE (solid) against epoch for both models. The faint
vertical line marks each run's selected checkpoint (`best_epoch`).

Reading it: train falling while val flattens/rises = overfitting; a train curve
stuck near 1.0 = that mode never fit (the broken `add` runs). Edit
`PLOT_CONFIGS` to choose which `(clr, tokenizer)` configs to draw. Keep raw and
CLR on SEPARATE plots for Model B - raw lives on the ~1e-4 simplex scale and
would flatten the CLR curves.

In [ ]:
import matplotlib.pyplot as plt

# (clr, tokenizer) configs to draw. Default = the CLR family (comparable scale).
PLOT_CONFIGS = [(True, m) for m in MODES]   # CLR x {scale, add, affine}

def _series(trainer, attr):
    recs = trainer.history.records
    return [r.epoch for r in recs], [getattr(r, attr) for r in recs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for ax, (model_name, idx) in zip(
        axes, [("Model A  micro->metab", 0), ("Model B  metab->micro", 1)]):
    for (clr, m) in PLOT_CONFIGS:
        tag = ("CLR " if clr else "raw ") + m
        tr = results[(clr, m)][idx]
        ep, trn = _series(tr, "train_loss")
        _,  val = _series(tr, "val_mse")
        line, = ax.plot(ep, val, lw=2, label=f"{tag} (val)")
        ax.plot(ep, trn, lw=1.3, ls="--", color=line.get_color(),
                label=f"{tag} (train)")
        be = tr.history.best_epoch
        if be is not None and be >= 0:
            ax.axvline(be, color=line.get_color(), alpha=0.25, lw=1)
    ax.set_title(model_name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Train (dashed) vs val (solid) MSE per epoch", y=1.02)
plt.tight_layout()
plt.show()

### How to read this

- **Did CLR help?** Compare `micro=CLR` vs `micro=raw` rows for the same model +
  tokenizer. For **Model B** the big tell is `sel_val_mse`: under `raw` it is the
  flat simplex floor (~0.0003, uninformative); under `CLR` it is ~unit-scale and
  actually tracks quality. `peak_r` for B should climb toward the **B CLR**
  ceiling (~0.21).
- **Model A** target is unchanged (standardized metabolome); CLR only changes its
  *input*. The FAIR bar is the CLR-input ridge (**A 0.15**, not 0.12): only
  `peak_r` clearly above ~0.15 means the transformer beats linear for A. Also
  watch `sel_val_mse` drop below the ~0.887 mean-fill floor.
- **`min_train_mse`** is the broken-mode check: a fusion that cannot drive
  training loss down is broken regardless of `peak_r`.
- Caveat: Model B's `peak_r` is measured in the target space (raw abundance vs
  CLR), so compare each against its OWN ceiling (raw ~0.14, CLR ~0.21).

Terminal equivalent for one config:
```
python run.py \
  --micro-emb embeddings/aligned/microbe_embeddings.csv \
  --metab-emb embeddings/aligned/metabolite_embeddings.csv \
  --max-epochs 60 --tokenizer scale --selection-metric val_mse --permutations 99
```
(CLR is on by default; add nothing to use it. Use the config flag
`microbiome_clr=False` to disable.)